<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Breakout/Large_Moves.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Resistance Breakout Strategy & Performance Scanner

This notebook implements a technical analysis scanner designed to identify **Resistance Breakouts**. The strategy focuses on identifying stocks that establish a clear price ceiling and subsequently break through it with high momentum.

**Key Features:**
* **Dynamic Resistance Detection:** Identifies price levels where a ticker has 'touched' or peaked multiple times (configurable via `min_touches`) within a lookback window.
* **Breakout Validation:** Triggers a signal only when the price closes above the resistance level by a specified percentage buffer and sets a new local high.
* **Ticker Locking:** Prevents redundant signals by enforcing a 'cool-off' period (e.g., 10 days) after a breakout is detected for a specific ticker.
* **Forward Performance Tracking:** Automatically calculates the 1-day through 5-day returns following each breakout and tracks the 'Success Rate' (percentage of time the price remains above the breakout level).
* **Buffered Data Ingestion:** Automatically downloads an extra 10 days of historical data beyond the scan window to ensure end-of-year signals have complete performance data.

**Current Configuration:** Scanning high-volume tickers for breakouts between January 2025 and December 2025.

In [15]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [16]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: raw_data
Deleted DataFrame: df_ticker
Deleted DataFrame: df_symbols
Deleted DataFrame: df_triggers
Deleted DataFrame: master_df
Deleted DataFrame: summary
Deleted DataFrame: df_3x
Deleted DataFrame: fwd_window
Deleted DataFrame: df_ret
Deleted DataFrame: summary_ret
All DataFrames cleared from memory.


In [17]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

In [24]:
# ---- USER PARAMETERS ----
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"
VOL_WINDOW = 60          # Rolling window for volatility
COOLDOWN_DAYS = 5        # Days to ignore new signals after a trigger
VOL_MULTIPLIERS = [1.5, 2.0, 2.5, 3.0, 4.0, 5.0]  # Multiples to test

# Calculate a warm-up start date (~90 calendar days to safely get 60 trading days)
start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
warmup_start_dt = start_dt - timedelta(days=90)
WARMUP_START_DATE = warmup_start_dt.strftime("%Y-%m-%d")

# ==========================================
# 2. LOAD TICKERS WITH GOOGLE DRIVE FALLBACK
# ==========================================
# URLs from your setup
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

# Default fallback list if Drive file is missing or broken
fallback_tickers = ["AAPL", "MSFT", "NVDA", "AMD", "TSLA"]

try:
    print("Attempting to pull tickers from the OptionVolume file...")
    df_symbols = pd.read_csv(OptionVolume)

    if 'Symbol' in df_symbols.columns:
        # Pull symbols, drop NaNs, convert to string, clean whitespace, and get uniques
        tickers = df_symbols['Symbol'].dropna().astype(str).unique().tolist()
        tickers = [t.strip() for t in tickers if t.strip()]
        print(f"Successfully loaded {len(tickers)} tickers from OptionVolume.")
    else:
        raise KeyError("'Symbol' column was not found in the downloaded file.")

except Exception as e:
    print(f"Failed to load tickers from Google Drive ({e}).")
    print("Falling back to the hardcoded ticker list.")
    tickers = fallback_tickers

print(f"Final ticker list size: {len(tickers)}")

# ==========================================
# 3. FETCH HISTORICAL DATA
# ==========================================
print(f"Fetching data from {WARMUP_START_DATE} to {END_DATE}...")
# Fetch data in bulk to minimize API calls
raw_data = yf.download(tickers, start=WARMUP_START_DATE, end=END_DATE, group_by='ticker')


Attempting to pull tickers from the OptionVolume file...


/tmp/ipykernel_541/3875655095.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(tickers, start=WARMUP_START_DATE, end=END_DATE, group_by='ticker')
[                       0%                       ]

Successfully loaded 100 tickers from OptionVolume.
Final ticker list size: 100
Fetching data from 2019-10-03 to 2025-12-31...


[*********************100%***********************]  100 of 100 completed


In [25]:
# ==========================================
# 4. SIGNAL PROCESSING ENGINE
# ==========================================

def process_signals(df, multiplier, direction='up'):
    """
    Identifies trigger days based on volatility multiples,
    enforces a cooldown period, and returns filtered rows.
    """
    signals = []
    cooldown_counter = 0

    for idx, row in df.iterrows():
        if cooldown_counter > 0:
            cooldown_counter -= 1
            continue

        # Check condition based on direction
        if direction == 'up' and row['Daily_Return'] > 0 and row['Return_Mag'] > row['Vol_Threshold']:
            signals.append(idx)
            cooldown_counter = COOLDOWN_DAYS
        elif direction == 'down' and row['Daily_Return'] < 0 and row['Return_Mag'] > row['Vol_Threshold']:
            signals.append(idx)
            cooldown_counter = COOLDOWN_DAYS

    return signals

# Master list to hold data from all stocks
all_results = []

# Dynamically ensure 4.0 and 5.0 are checked if not already defined globally
extended_multipliers = list(set(VOL_MULTIPLIERS + [4.0, 5.0]))
extended_multipliers.sort()

for ticker in tickers:
    if ticker not in raw_data.columns.levels[0]:
        continue

    # Extract ticker data and drop completely missing rows
    df_ticker = raw_data[ticker].dropna(subset=['Close']).copy()
    if len(df_ticker) < VOL_WINDOW:
        continue

    # Calculate Daily Returns
    df_ticker['Daily_Return'] = df_ticker['Close'].pct_change()
    df_ticker['Return_Mag'] = df_ticker['Daily_Return'].abs()

    # NEW LOGIC: Store the previous day's close directly on the row to capture full impulse gaps
    df_ticker['Prev_Close'] = df_ticker['Close'].shift(1)

    # Calculate Historic Volatility (Rolling Std Dev of Daily Returns)
    df_ticker['Hist_Vol'] = df_ticker['Daily_Return'].rolling(window=VOL_WINDOW).std()

    # Slice data to drop the warm-up period
    df_ticker = df_ticker.loc[START_DATE:]
    if df_ticker.empty:
        continue

    # Calculate Forward Returns (1 to 5 days out)
    for i in range(1, 6):
        df_ticker[f'Fwd_Ret_{i}d'] = df_ticker['Close'].shift(-i) / df_ticker['Close'] - 1

    # Loop through multipliers (including 4.0x and 5.0x) and directions to isolate signals
    for mult in extended_multipliers:
        df_ticker['Vol_Threshold'] = df_ticker['Hist_Vol'] * mult

        for direction in ['up', 'down']:
            trigger_indices = process_signals(df_ticker, mult, direction=direction)

            if trigger_indices:
                df_triggers = df_ticker.loc[trigger_indices].copy()
                df_triggers['Ticker'] = ticker
                df_triggers['Multiplier'] = mult
                df_triggers['Direction'] = direction

                # Keep relevant columns, including 'Prev_Close' for the downstream retracement engine
                cols_to_keep = ['Ticker', 'Multiplier', 'Direction', 'Daily_Return', 'Hist_Vol', 'Prev_Close', 'Close',
                                'Fwd_Ret_1d', 'Fwd_Ret_2d', 'Fwd_Ret_3d', 'Fwd_Ret_4d', 'Fwd_Ret_5d']
                all_results.append(df_triggers[cols_to_keep])

# Combine all results into a single Master DataFrame
if all_results:
    master_df = pd.concat(all_results)
    print(f"Successfully processed signals. Total triggers found: {len(master_df)}")
else:
    master_df = pd.DataFrame()
    print("No signals found with the given parameters.")

Successfully processed signals. Total triggers found: 23193


In [26]:
if not master_df.empty:
    # Target columns to average out
    fwd_return_cols = ['Fwd_Ret_1d', 'Fwd_Ret_2d', 'Fwd_Ret_3d', 'Fwd_Ret_4d', 'Fwd_Ret_5d']

    # Group by Multiplier and Direction, calculating the average return and total signal count
    summary = master_df.groupby(['Multiplier', 'Direction']).agg(
        Signal_Count=('Daily_Return', 'count'),
        Avg_Fwd_1d=('Fwd_Ret_1d', 'mean'),
        Avg_Fwd_2d=('Fwd_Ret_2d', 'mean'),
        Avg_Fwd_3d=('Fwd_Ret_3d', 'mean'),
        Avg_Fwd_4d=('Fwd_Ret_4d', 'mean'),
        Avg_Fwd_5d=('Fwd_Ret_5d', 'mean')
    )

    # Format as percentages for cleaner reading
    print(summary.to_string(formatters={c: '{:,.2%}'.format for c in summary.columns if 'Avg' in c}))

                      Signal_Count Avg_Fwd_1d Avg_Fwd_2d Avg_Fwd_3d Avg_Fwd_4d Avg_Fwd_5d
Multiplier Direction                                                                     
1.5        down               5320     -0.03%     -0.11%     -0.11%      0.24%      0.21%
           up                 6194      0.04%      0.26%      0.51%      0.49%      0.65%
2.0        down               2769      0.05%     -0.23%     -0.25%      0.31%      0.13%
           up                 3161      0.04%      0.33%      0.47%      0.47%      0.57%
2.5        down               1385      0.05%     -0.52%     -0.56%      0.34%     -0.02%
           up                 1695     -0.10%      0.36%      0.75%      0.52%      0.59%
3.0        down                758      0.21%     -0.86%     -1.25%      0.13%     -0.63%
           up                  987     -0.19%      0.40%      0.87%      0.53%      0.43%
4.0        down                286      0.73%     -0.73%     -0.67%      0.58%     -0.31%
          

In [30]:
# ==========================================
# 5. PARAMETERIZED DAY-BY-DAY CLOSING RETRACEMENT ENGINE
# ==========================================

# ---- TARGET MULTIPLIER PARAMETER ----
TARGET_MULTIPLIER = 3.0  # <--- Change this to 4.0 or 5.0 to analyze different extremes

if 'master_df' in globals() and not master_df.empty:
    # Filter master dataframe for the chosen multiplier signals
    df_filtered = master_df[master_df['Multiplier'] == TARGET_MULTIPLIER].copy()

    # Master list to hold our daily retracement records
    daily_retracement_records = []

    for idx, trigger in df_filtered.iterrows():
        ticker = trigger['Ticker']
        direction = trigger['Direction']

        # Get the raw historical data for this ticker
        df_ticker = raw_data[ticker].dropna(subset=['Close'])

        try:
            trigger_loc = df_ticker.index.get_loc(idx)
            # Ensure we have 5 days of forward data available to look at
            if trigger_loc + 5 >= len(df_ticker):
                continue

            # Day 0 Close and Previous Close
            day_0_close = df_ticker.iloc[trigger_loc]['Close']
            prev_day_close = df_ticker.iloc[trigger_loc - 1]['Close']

            # Calculate the size/magnitude of the total move (including overnight gaps)
            impulse_size = day_0_close - prev_day_close
            if impulse_size == 0:
                continue

        except (KeyError, IndexError):
            continue

        # Calculate exact target price boundaries based on Previous Close baseline
        if direction == 'up':
            target_50  = day_0_close - (0.50 * impulse_size)
            target_75  = day_0_close - (0.75 * impulse_size)
            target_100 = day_0_close - (1.00 * impulse_size)
        else: # down
            abs_imp_size = abs(impulse_size)
            target_50  = day_0_close + (0.50 * abs_imp_size)
            target_75  = day_0_close + (0.75 * abs_imp_size)
            target_100 = day_0_close + (1.00 * abs_imp_size)

        # Check each day from T+1 to T+5 individually
        for d in range(1, 6):
            fwd_close = df_ticker.iloc[trigger_loc + d]['Close']

            if direction == 'up':
                hit_50  = int(fwd_close <= target_50)
                hit_75  = int(fwd_close <= target_75)
                hit_100 = int(fwd_close <= target_100)
            else: # direction == 'down'
                hit_50  = int(fwd_close >= target_50)
                hit_75  = int(fwd_close >= target_75)
                hit_100 = int(fwd_close >= target_100)

            daily_retracement_records.append({
                'Direction': direction,
                'Day': d,
                'Hit_50': hit_50,
                'Hit_75': hit_75,
                'Hit_100': hit_100
            })

    # Convert to DataFrame and aggregate
    if daily_retracement_records:
        df_daily_ret = pd.DataFrame(daily_retracement_records)

        # Group by Direction and Day to calculate the probabilities and count
        summary_daily = df_daily_ret.groupby(['Direction', 'Day']).agg(
            Total_Signals=('Hit_50', 'count'),
            Retrace_50_Pct=('Hit_50', 'mean'),
            Retrace_75_Pct=('Hit_75', 'mean'),
            Retrace_100_Pct=('Hit_100', 'mean')
        )

        print(f"\n=== DAY-BY-DAY CLOSING RETRACEMENT PROBABILITIES ({TARGET_MULTIPLIER}x MULTIPLIER) ===")
        pct_cols = ['Retrace_50_Pct', 'Retrace_75_Pct', 'Retrace_100_Pct']
        print(summary_daily.to_string(formatters={c: '{:,.2%}'.format for c in pct_cols}))
    else:
        print(f"No valid {TARGET_MULTIPLIER}x multiplier signals available for daily tracking.")
else:
    print("Master DataFrame is empty or not found. Please run the signal processing engine first.")


=== DAY-BY-DAY CLOSING RETRACEMENT PROBABILITIES (3.0x MULTIPLIER) ===
               Total_Signals Retrace_50_Pct Retrace_75_Pct Retrace_100_Pct
Direction Day                                                             
down      1              758         10.95%          3.56%           1.32%
          2              758          8.97%          3.83%           1.32%
          3              758         13.32%          6.07%           2.77%
          4              758         17.02%         10.03%           5.28%
          5              758         17.28%         10.42%           6.33%
up        1              986         11.05%          5.27%           2.94%
          2              986         11.87%          6.09%           2.74%
          3              986         16.02%          8.82%           5.58%
          4              986         18.97%         11.56%           7.10%
          5              986         22.62%         14.00%           9.33%
